### Implement Kmeans from Scratch

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import plotly.graph_objects as go

##### Data Loading and Analysis

In [4]:
iris = pd.read_csv("./data/Iris.csv")
iris.drop('Id', inplace=True, axis = 1)

In [ ]:
X = iris.iloc[:, : -1] # Set our training data

y = iris.iloc[:, -1]

In [6]:
iris.head().style.background_gradient(cmap=sns.cubehelix_palette(as_cmap=True))

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.100000,3.500000,1.400000,0.200000,Iris-setosa
1,4.900000,3.000000,1.400000,0.200000,Iris-setosa
2,4.700000,3.200000,1.300000,0.200000,Iris-setosa
3,4.600000,3.100000,1.500000,0.200000,Iris-setosa
4,5.000000,3.600000,1.400000,0.200000,Iris-setosa


##### Data Distribution 

In [7]:
fig = px.pie(iris, 'Species', 
             color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'],
             title='Data Distribution',template='plotly')

fig.show()

##### From this plot, we conclude that:
The data is perfectly balanced.

##### Sepal-Length

In [9]:
fig = px.box(data_frame=iris, x='Species', y='SepalLengthCm', color='Species', 
             color_discrete_sequence=['#29066B','#7D3AC1','#EB548C'], orientation='v')

fig.show()

In [10]:
fig = px.histogram(data_frame=iris, x='SepalLengthCm',color='Species',color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'],nbins=50)
fig.show()

From these plots we conclude that:¶
- Setosa has much smaller SepalLength than the other 2 classes

- Virginca has the highest SepalLength, however It seems hard to distingush between Virginca and Versicolor using SepalLength as the difference is less clear

- We can see that Virginica contains an outlier

##### SepalWidth

In [11]:
fig = px.box(data_frame=iris, x='Species',y='SepalWidthCm',color='Species',color_discrete_sequence=['#29066B','#7D3AC1','#EB548C'],orientation='v')
fig.show()

In [12]:
fig = px.histogram(data_frame=iris, x='SepalWidthCm',color='Species',color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'],nbins=30)
fig.show()

From these plots we conclude that:
- Setosa has larger SepalWidth than the other 2 classes

- Versicolo has smaller SepalWidth than the other 2 classes

- Overall all classes seem to have relatively close value of sepalwidth which indicate that is might not be a very useful feature

##### Petal-Length

In [13]:
fig = px.box(data_frame=iris, x='Species',y='PetalLengthCm',color='Species',color_discrete_sequence=['#29066B','#7D3AC1','#EB548C'],orientation='v')
fig.show()

In [14]:
fig = px.histogram(data_frame=iris, x='PetalLengthCm',color='Species',color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'],nbins=30)
fig.show()

From these plots we conclude that:
- Setosa has much smaller PetaLength than the other 2 classes

- This difference is less clear between Virginica and Versicolor

- Overall this seems like an PetaLength interesting feature

##### Petal-Width

In [15]:
fig = px.box(data_frame=iris, x='Species',y='PetalWidthCm',color='Species',color_discrete_sequence=['#29066B','#7D3AC1','#EB548C'],orientation='v')
fig.show()

In [16]:
fig = px.histogram(data_frame=iris, x='PetalWidthCm',color='Species',color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'],nbins=30)
fig.show()

From these plots we conclude that:
- Setosa has much smaller PetalWidth than the other 2 classes

- This difference is less clear between Virginica and Versicolor

- Overall this seems like an PetalWidth interesting feature

In [17]:
fig = px.scatter(data_frame=iris, x='SepalLengthCm',y='SepalWidthCm'
           ,color='Species',size='PetalLengthCm',template='seaborn',color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'],)

fig.update_layout(width=800, height=600,
                  xaxis=dict(color="#BF40BF"),
                 yaxis=dict(color="#BF40BF"))
fig.show()

In [18]:
fig = px.scatter(data_frame=iris, x='PetalLengthCm',y='PetalWidthCm'
           ,color='Species',size='SepalLengthCm',template='seaborn',color_discrete_sequence=['#491D8B','#7D3AC1','#EB548C'],)

fig.update_layout(width=800, height=600,
                  xaxis=dict(color="#BF40BF"),
                 yaxis=dict(color="#BF40BF"))
fig.show()

### Model Implementation

#### K-Means Clustering
k means clustering is a method used to group data points into clusters. It operates through an interactive process as follows:

##### Initialization:
- We begin by randomly selecting initial cluster centroids. These centroids are points inour dataset that will act as the centers of our clusters

##### Assignment of Points to Centroids
- For each data point in our training set, we determine which centroid it is closest to. This is done by measuring the distance between the data point and each centroid and selecting the centroid with the smallest distance as the closest one. We use an index notation, denoted as $c^{(i)}$, to represent the index of the closest centroid to the data point $x^{(i)}$

##### Computing Cluster Means
- After assigning data points to their closest centroids, we calculate the mean of all data points within each cluster. This mean becomes the new centroid for thata cluster.
The goal of K-means is to minimize a cost function, which is typically the squared Euclidean distance between each data point and its assigned centroid:
$$\sum_{i = 1}^{m} ||x^{(i)} - \mu_{j}||^2$$
**Where**
- $m$ is the number of data points.
- $x^{(i)}$ represents the i-th data point.
- $\mu_{j}$ **represents the centroid of the j-th cluster**

The algorithm repeats the assignment and recomputation of centroids until convergence, where the centroids no longer change significantly or a specified number of iterations is reached. 

In [ ]:
# This code is here to help you understand how numpy broadcasting works. 

samples = np.array([[1, 2, 3], [2, 4, 5], [6, 8, 9], [1, 4, 7], [9, 0, 5], [3, 4, 7]])
centroids = np.array([[1, 4, 7], [3, 4, 7]])
samples = np.expand_dims(samples, axis = 1)  # it wouldn't work if you remove this line.
# print(samples)

# !!!! Keep in mind that broadcasting and matching in python goes from right to left.
# A:  ( n_samples , k , n_features )
# B:            ( k , n_features )
#       ───────────────▶ compare


print(samples - centroids)
# as you may see, we have successfully obtained a structure where the each data sample gets to 
# subtract each centroid.

[[[ 0 -2 -4]
  [-2 -2 -4]]

 [[ 1  0 -2]
  [-1  0 -2]]

 [[ 5  4  2]
  [ 3  4  2]]

 [[ 0  0  0]
  [-2  0  0]]

 [[ 8 -4 -2]
  [ 6 -4 -2]]

 [[ 2  0  0]
  [ 0  0  0]]]


In [43]:
class Kmeans:
    """
        K-Means clustering algorithm implementation.

        Parameters:
            K (int): Number of clusters
        
        Attributes:
            K (int): Number of clusers
            centroids (numpy.ndarray): Array containing the centroids of each cluster
        
        Methods: 
            __init__(self, K): Initializes the Kmeans instance with the specified number of clusters.
            initialize_centroids(self, X): Initializes the centroids for each cluster by selecting K random points from the dataset.
            assign_points_centroids(self, X): Assigns each point in the dataset to the nearest centroid.
            compute_mean(self, X, points): Computes the mean of the points assigned to each centroid.
            fit(self, X, iterations=10): Clusters the dataset using the K-Means algorithm.
    """

    def __init__(self, K):
        assert K > 0, "K should be a positive integer."
        self.K = K

    def initialize_centroids(self, X):
        assert X.shape[0] >= self.K, "Number of data points should be greater than or equal to K."

        randomized_X = np.random.permutation(X.shape[0])
        centroid_idx = randomized_X[:self.K]  # picking a random subset from training data as the centroid.

        self.centroids = X[centroid_idx] # assign the centroids to the selected points
    
    def assign_points_centroids(self, X):
        X = np.expand_dims(X, axis = 1)
        distance = np.linalg.norm((X - self.centroids), axis = -1)
        points = np.argmin(distance, axis = 1)  # assign each point to the closest centroid.

        assert len(points) == X.shape[0], "Number of assigned points should equal the number of data points"
        return points

    def compute_mean(self, X, points):
        """
            Compute the mean of the points assigned to each centroid.

            Parameters:
                X (numpy.ndarray): dataset to cluster
                points (numpy.ndarray): array containing the index of the centroid for each point.
            
            Returns:
                numpy.ndarray: array containing the new centroids for each cluster.
        """
        centroids = np.zeros((self.K, X.shape[1])) # initialize arrat to store centroids.
        # shape of centroids is (# of centroids, # of features)

        for i in range(self.K):
            centroid_mean = X[points == i].mean(axis = 0) 
            centroids[i] = centroid_mean

        return centroids
    
    def fit(self, X, iterations = 10):
        """
        Cluster the dataset using the K-Means algorithm.

        Parameters:
            X (numpy.ndarray): dataset to cluster
            iterations (int): number of iterations to perform (default = 10)

        Returns:
            numpy.ndarray: array containing the final centroids for each cluster
            numpy.ndarray: array containing the index of the centorid for each point.
        """
        self.initialize_centroids(X) # initialize the centroids.
        for i in range(iterations):
            points = self.assign_points_centroids(X)
            self.centroids = self.compute_mean(X, points)

            # Assertions for debugging and validation
            assert len(self.centroids) == self.K, "Number of centroids should equal K."
            assert X.shape[1] == self.centroids.shape[1], "Dimensionality of centroids should match input data."
            assert max(points) < self.K, "Cluster index should be less than K"
            assert min(points) >= 0, "Cluster index should be non-negative"

        return self.centroids, points


Example to help us understand what the function expand_dims does

In [44]:
x = np.array([[1,2, 3],
              [4, 5, 6],
              [7, 8, 9]])
x1 = np.expand_dims(x, axis =2)


y = np.array([1, 2, 3])
y1 = np.expand_dims(y, axis = 1)

## Depending on the layer of dimesion our numpy array has, we increase the values from 0 to n - 1, 
## Starting from the outermost to the innermost with the increment of n.
y1

array([[1],
       [2],
       [3]])

In [84]:
kmeans = Kmeans(3)
centroids, points = kmeans.fit(X, 1000)

print (X[points == 1, 0])
# print(centroids)
# centroids[:, 0]  # first column

[5.1 5.  5.4 5.  5.4 5.8 5.7 5.4 5.1 5.7 5.1 5.4 5.1 4.6 5.1 5.  5.2 5.2
 5.4 5.2 5.5 5.  5.5 5.1 5.  5.  5.1 5.1 5.3 5. ]


#### Evaluation
**Visualize the result**

In [51]:
fig  = go.Figure()
fig.add_trace(go.Scatter(
    x=X[points ==0, 0], y = X[points==0, 1],
    mode= 'markers', marker_color = '#DB4CB2', name="Iris-setosa"
))
fig.add_trace(go.Scatter(
    x=X[points == 1, 0], y=X[points == 1, 1],
    mode='markers',marker_color='#c9e9f6',name='Iris-versicolour'
))

fig.add_trace(go.Scatter(
    x=X[points == 2, 0], y=X[points == 2, 1],
    mode='markers',marker_color='#7D3AC1',name='Iris-virginica'
))

fig.add_trace(go.Scatter(
    x=centroids[:, 0], y=centroids[:,1],
    mode='markers',marker_color='#CAC9CD',marker_symbol=4,marker_size=13,name='Centroids'
))
fig.update_layout(template='plotly_dark',width=1000, height=500,)

### Implement KMeans using Scikit Learn

In [85]:
from sklearn.cluster import KMeans

In [86]:
model_kmeans = KMeans(3)
model_kmeans.fit(X)

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",3
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",None
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'
